# Treinamento de Modelos ML XGBoost e CatBoost

Este notebook executa o fluxo completo de treinamento, otimização e validação de modelos baseados em árvore.

### Importação das Bibliotecas
Importa as dependências do ecossistema Scikit-Learn, Optuna para otimização e as bibliotecas base.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs('models/result_pkl', exist_ok=True)
os.makedirs('resultados', exist_ok=True)

## 1. Funções Utilitárias

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def evaluate(y_true, y_pred):
    y_true, y_pred = np.array(y_true).flatten(), np.array(y_pred).flatten()
    mae = np.mean(np.abs(y_true - y_pred))
    r = rmse(y_true, y_pred)
    mean_y = np.mean(y_true)
    nrmse = r / mean_y if mean_y != 0 else 0
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))
    return {'MAE': round(mae, 2), 'RMSE': round(r, 2), 'nRMSE': round(nrmse, 4), 'sMAPE': round(smape, 2)}

### Preparação e Tratamento de Nulos
Isola-se a variável alvo (`Revenue`) e aplica-se o Forward Fill seguido de Backward Fill temporal para preencher lacunas sem causar vazamento de dados do futuro.

In [ ]:
def load_data(csv_path):
    df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
    target = 'Revenue'
    unavailable_cols = {'UnitPrice_median', 'UnitPrice_std', 'UniqueCustomers', 'UniqueProducts'}
    feature_cols = [
        c for c in df.columns
        if c not in ['Date', target]
        and pd.api.types.is_numeric_dtype(df[c])
        and not c.startswith(('RevenueLag_', 'Rolling'))
        and c not in unavailable_cols
    ]

    cutoff = pd.Timestamp('2011-09-01')
    train_df = df.loc[df['Date'] < cutoff].reset_index(drop=True)
    test_df = df.loc[df['Date'] >= cutoff].reset_index(drop=True)

    return train_df, test_df, feature_cols


def prepare_fold_features(train_df, apply_df):
    frame = apply_df.copy()

    overall_mean = train_df['Revenue'].mean()
    overall_median = train_df['Revenue'].median()
    overall_q25 = train_df['Revenue'].quantile(0.25)
    overall_q75 = train_df['Revenue'].quantile(0.75)
    overall_iqr = overall_q75 - overall_q25
    overall_std = train_df['Revenue'].std()
    if pd.isna(overall_std):
        overall_std = 0.0

    group_defaults = {
        'Weekday': {'mean': overall_mean, 'median': overall_median, 'IQR': overall_iqr},
        'Month': {'mean': overall_mean, 'median': overall_median, 'IQR': overall_iqr},
        'Quarter': {'mean': overall_mean, 'median': overall_median, 'IQR': overall_iqr},
    }

    for col, defaults in group_defaults.items():
        stats = train_df.groupby(col)['Revenue'].agg(
            mean='mean',
            median='median',
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
        )
        stats['IQR'] = stats['q75'] - stats['q25']
        frame[f'{col}Revenue_mean'] = frame[col].map(stats['mean']).fillna(defaults['mean'])
        frame[f'{col}Revenue_median'] = frame[col].map(stats['median']).fillna(defaults['median'])
        frame[f'{col}Revenue_IQR'] = frame[col].map(stats['IQR']).fillna(defaults['IQR'])

    xmas_stats = train_df.groupby('PreChristmas')['Revenue'].agg(
        PreChristmasRevenue_mean='mean',
        PreChristmasRevenue_std='std'
    )
    frame['PreChristmasRevenue_mean'] = frame['PreChristmas'].map(xmas_stats['PreChristmasRevenue_mean']).fillna(overall_mean)
    frame['PreChristmasRevenue_std'] = frame['PreChristmas'].map(xmas_stats['PreChristmasRevenue_std']).fillna(overall_std)
    frame['PreChristmasRevenue_std'] = frame['PreChristmasRevenue_std'].fillna(0.0)

    return frame

## 2. Carregamento dos Dados

### Carregamento das Partições
Lê as três frentes de dados independentes (Produto, País, Cliente) e cria as matrizes de características (`X`) e a série alvo (`y`).

In [ ]:
datasets = {
    'Produto': 'data/product.csv',
    'País':    'data/country.csv',
    'Cliente': 'data/customer.csv'
}

data = {}
for name, path in datasets.items():
    train_df, test_df, feat_cols = load_data(path)
    data[name] = {'train_df': train_df, 'test_df': test_df, 'feature_cols': feat_cols}
    print(f"  {name}: train_samples={len(train_df)}, test_samples={len(test_df)}")

## 3. XGBoost (5-Fold Time Series Split)

### Otimização e Treinamento do XGBoost
Aplica validação cruzada (`TimeSeriesSplit` com 5 Folds). O Optuna utiliza os últimos 15% do treino atual como validação interna para achar hiperparâmetros ótimos.

**Correções aplicadas:**
- `best_params_global`: rastreia, ao longo de todos os folds, os hiperparâmetros cujo `study.best_value` (RMSE interno no Optuna) foi o menor — não mais os parâmetros do último fold.
- `MAE_Test` / `RMSE_Test`: calculados sobre o conjunto de **teste real** (futuro), após treinar o modelo final com todo o conjunto de treino.
- `MAE_CV` / `RMSE_CV`: mantidos como diagnóstico da estabilidade temporal, mas **não** usados como métrica principal.

In [ ]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

results_xgb = {}
for name, d in data.items():
    print(f"\nOtimizando XGBoost para a serie: {name}")
    train_df, test_df, feature_cols = d['train_df'], d['test_df'], d['feature_cols']

    def objective_xgb(trial):
        params = {
            'n_estimators': 300,
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'random_state': 42,
            'verbosity': 0
        }
        fold_maes = []
        fold_rmses = []
        for train_idx, val_idx in tscv.split(train_df):
            fold_train_df = train_df.iloc[train_idx].copy()
            fold_val_df = train_df.iloc[val_idx].copy()

            fold_train_features = prepare_fold_features(fold_train_df, fold_train_df)
            fold_val_features = prepare_fold_features(fold_train_df, fold_val_df)

            X_fold_train = fold_train_features[feature_cols].values
            y_fold_train = fold_train_features['Revenue'].values
            X_fold_val = fold_val_features[feature_cols].values
            y_fold_val = fold_val_features['Revenue'].values

            inner_val_size = max(int(len(X_fold_train) * 0.15), 1)
            X_t_inner, y_t_inner = X_fold_train[:-inner_val_size], y_fold_train[:-inner_val_size]
            X_v_inner, y_v_inner = X_fold_train[-inner_val_size:], y_fold_train[-inner_val_size:]

            model = XGBRegressor(**params, early_stopping_rounds=20)
            model.fit(
                X_t_inner,
                np.log1p(y_t_inner),
                eval_set=[(X_v_inner, np.log1p(y_v_inner))],
                verbose=False
            )

            preds = np.expm1(model.predict(X_fold_val))
            fold_maes.append(np.mean(np.abs(y_fold_val - preds)))
            fold_rmses.append(rmse(y_fold_val, preds))

        mean_mae = float(np.mean(fold_maes)) if fold_maes else float('inf')
        mean_rmse = float(np.mean(fold_rmses)) if fold_rmses else float('inf')
        trial.set_user_attr('mae_mean', mean_mae)
        trial.set_user_attr('rmse_mean', mean_rmse)
        return mean_rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective_xgb, n_trials=15)

    best_trial = study.best_trial
    best_params = {**best_trial.params, 'n_estimators': 300, 'random_state': 42, 'verbosity': 0}

    full_train_features = prepare_fold_features(train_df, train_df)
    test_features = prepare_fold_features(train_df, test_df)
    X_train = full_train_features[feature_cols].values
    y_train = full_train_features['Revenue'].values
    X_test = test_features[feature_cols].values
    y_test = test_features['Revenue'].values

    final_model = XGBRegressor(**best_params)
    final_model.fit(X_train, np.log1p(y_train), verbose=False)
    test_preds = np.expm1(final_model.predict(X_test))

    test_metrics = evaluate(y_test, test_preds)
    joblib.dump(final_model, f'models/result_pkl/xgb_{name}.pkl')

    results_xgb[name] = {
        'MAE_CV': round(best_trial.user_attrs['mae_mean'], 2),
        'RMSE_CV': round(best_trial.user_attrs['rmse_mean'], 2),
        'MAE_Test': test_metrics['MAE'],
        'RMSE_Test': test_metrics['RMSE'],
        'nRMSE_Test': test_metrics['nRMSE'],
        'sMAPE_Test': test_metrics['sMAPE'],
        'periodos_previstos': len(y_test),
        'valores_previstos': test_preds.tolist()
    }

## 4. CatBoost (5-Fold Time Series Split)

### Otimização e Treinamento do CatBoost
Semelhante ao XGBoost — mesmas correções aplicadas: seleção global de hiperparâmetros e métricas de teste reais.

In [ ]:
results_cat = {}
for name, d in data.items():
    print(f"\nOtimizando CatBoost para a serie: {name}")
    train_df, test_df, feature_cols = d['train_df'], d['test_df'], d['feature_cols']

    def objective_cat(trial):
        params = {
            'iterations': 300,
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 6),
            'random_seed': 42,
            'verbose': 0
        }
        fold_maes = []
        fold_rmses = []
        for train_idx, val_idx in tscv.split(train_df):
            fold_train_df = train_df.iloc[train_idx].copy()
            fold_val_df = train_df.iloc[val_idx].copy()

            fold_train_features = prepare_fold_features(fold_train_df, fold_train_df)
            fold_val_features = prepare_fold_features(fold_train_df, fold_val_df)

            X_fold_train = fold_train_features[feature_cols].values
            y_fold_train = fold_train_features['Revenue'].values
            X_fold_val = fold_val_features[feature_cols].values
            y_fold_val = fold_val_features['Revenue'].values

            inner_val_size = max(int(len(X_fold_train) * 0.15), 1)
            X_t_inner, y_t_inner = X_fold_train[:-inner_val_size], y_fold_train[:-inner_val_size]
            X_v_inner, y_v_inner = X_fold_train[-inner_val_size:], y_fold_train[-inner_val_size:]

            model = CatBoostRegressor(**params)
            model.fit(
                X_t_inner,
                np.log1p(y_t_inner),
                eval_set=(X_v_inner, np.log1p(y_v_inner)),
                early_stopping_rounds=20,
                verbose=False
            )

            preds = np.expm1(model.predict(X_fold_val))
            fold_maes.append(np.mean(np.abs(y_fold_val - preds)))
            fold_rmses.append(rmse(y_fold_val, preds))

        mean_mae = float(np.mean(fold_maes)) if fold_maes else float('inf')
        mean_rmse = float(np.mean(fold_rmses)) if fold_rmses else float('inf')
        trial.set_user_attr('mae_mean', mean_mae)
        trial.set_user_attr('rmse_mean', mean_rmse)
        return mean_rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective_cat, n_trials=15)

    best_trial = study.best_trial
    best_params = {**best_trial.params, 'iterations': 300, 'random_seed': 42, 'verbose': 0}

    full_train_features = prepare_fold_features(train_df, train_df)
    test_features = prepare_fold_features(train_df, test_df)
    X_train = full_train_features[feature_cols].values
    y_train = full_train_features['Revenue'].values
    X_test = test_features[feature_cols].values
    y_test = test_features['Revenue'].values

    final_model = CatBoostRegressor(**best_params)
    final_model.fit(X_train, np.log1p(y_train), verbose=False)
    test_preds = np.expm1(final_model.predict(X_test))

    test_metrics = evaluate(y_test, test_preds)
    joblib.dump(final_model, f'models/result_pkl/cat_{name}.pkl')

    results_cat[name] = {
        'MAE_CV': round(best_trial.user_attrs['mae_mean'], 2),
        'RMSE_CV': round(best_trial.user_attrs['rmse_mean'], 2),
        'MAE_Test': test_metrics['MAE'],
        'RMSE_Test': test_metrics['RMSE'],
        'nRMSE_Test': test_metrics['nRMSE'],
        'sMAPE_Test': test_metrics['sMAPE'],
        'periodos_previstos': len(y_test),
        'valores_previstos': test_preds.tolist()
    }

### Exportação dos Resultados
Consolida as métricas de CV (diagnóstico de estabilidade) e de **teste real** (desempenho no futuro) para todas as séries, junto com os valores previstos, em um arquivo CSV final.

In [ ]:
import os
os.makedirs('resultados', exist_ok=True)

rows = []
for name, m in results_xgb.items():
    rows.append({
        'serie':              name,
        'periodos_previstos': m['periodos_previstos'],
        'modelo':             'XGBoost',
        'valores_previstos':  m['valores_previstos'],
        'MAE_CV':             round(m['MAE_CV'],    2),
        'RMSE_CV':            round(m['RMSE_CV'],   2),
        'MAE_Test':           round(m['MAE_Test'],  2),
        'RMSE_Test':          round(m['RMSE_Test'], 2),
    })
for name, m in results_cat.items():
    rows.append({
        'serie':              name,
        'periodos_previstos': m['periodos_previstos'],
        'modelo':             'CatBoost',
        'valores_previstos':  m['valores_previstos'],
        'MAE_CV':             round(m['MAE_CV'],    2),
        'RMSE_CV':            round(m['RMSE_CV'],   2),
        'MAE_Test':           round(m['MAE_Test'],  2),
        'RMSE_Test':          round(m['RMSE_Test'], 2),
    })

df_results = pd.DataFrame(rows)
df_results.to_csv('resultados/resultados_ML.csv', index=False)
print('Resultados salvos em resultados/resultados_ML.csv')
df_results